In [1]:
library(vegan)
library(tidyverse)
library(pairwiseAdonis)
library(emmeans)

Loading required package: permute

Warning message:
“Your system is mis-configured: ‘/etc/localtime’ is not a symlink”
Warning message:
“It is strongly recommended to set envionment variable TZ to ‘America/Sao_Paulo’ (or equivalent)”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: cluster

Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'



In [2]:
dados <- read.csv('../dados/descritores_v2.csv')

### Auxiliar

In [3]:
descritores <- c("centroide", "logattacktime")
fatores <- c("material", "nota", "metodo")

In [4]:
set.seed(20260601)

In [5]:
printCabecalho <- function(texto, largura = 70) {
  cat("\n", rep("=", largura), "\n", sep = "")
  cat(texto, "\n")
  cat(rep("=", largura), "\n\n", sep = "")
}

In [6]:
prepDados <- function(df, descritores) {
  dados_padronizados <- df %>%
    select(all_of(descritores)) %>%
    scale() %>%
    as.data.frame()
  
  return(dados_padronizados)
}

# PERMANOVA

In [7]:
calcMatrizDistancia <- function(dados, metodo = "euclidean") {
  if (is.data.frame(dados) && ncol(dados) > 1) {
    return(vegdist(dados, method = metodo))
  }
  return(dist(dados))
}

In [8]:
execPermanova <- function(matriz_dist, dados, fatores, permutations, by) {
  termos <- paste(fatores, collapse = " * ")
  formula <- as.formula(paste("matriz_dist ~", termos))
  
  modelo <- adonis2(
    formula,
    data = dados,
    permutations = permutations,
    by = by
  )
  
  return(modelo)
}

In [9]:
analiseUnivariada <- function(df, descritor, fatores, permutations, by) {
  printCabecalho(paste(toupper(descritor)))
  
  dados <- data.frame(
    descritor_valor = scale(df[[descritor]]),
    df[fatores]
  )

  matriz_dist <- calcMatrizDistancia(dados$descritor_valor)
  modelo <- execPermanova(matriz_dist, dados, fatores, permutations, by = by)
  
  print(modelo)
}


In [10]:
analiseMultivariada <- function(df, descritores, fatores, permutations, by) {
  titulo <- paste(paste(toupper(descritores), collapse = " + "))
  printCabecalho(titulo)
  
  dados_combinados <- prepDados(df, descritores)
  matriz_dist <- calcMatrizDistancia(dados_combinados, "euclidean")
  modelo <- execPermanova(matriz_dist, df, fatores, permutations, by = by)
  
  print(modelo)
}


In [11]:
PERMANOVA <- function(df, descritores, fatores, permutations = 9999, by) {

  
  # Análises individuais
  for (descritor in descritores) {
    modelo <- analiseUnivariada(df, descritor, fatores, permutations, by = by)
  }
  
  # Análise combinada
  modelo_combinado <- analiseMultivariada(df, descritores, fatores, permutations, by = by)
}


## Univariada e multivariada

In [12]:
PERMANOVA(
  df = dados,
  descritores = descritores,
  fatores = fatores,
  permutations = 9999,
  by = 'term'
)


CENTROIDE 

Permutation test for adonis under reduced model
Terms added sequentially (first to last)
Permutation: free
Number of permutations: 9999

adonis2(formula = formula, data = dados, permutations = permutations, by = by)
                     Df SumOfSqs      R2          F Pr(>F)    
material              1    0.023 0.00029     3.9981 0.0486 *  
nota                  3    2.950 0.03734   170.0408 0.0001 ***
metodo                1   70.155 0.88804 12133.3257 0.0001 ***
material:nota         3    0.071 0.00090     4.1127 0.0098 ** 
material:metodo       1    0.003 0.00004     0.5010 0.4838    
nota:metodo           3    5.350 0.06772   308.4416 0.0001 ***
material:nota:metodo  3    0.078 0.00098     4.4816 0.0072 ** 
Residual             64    0.370 0.00468                      
Total                79   79.000 1.00000                      
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

LOGATTACKTIME 

Permutation test for adonis under reduced model
Terms add

# Global

In [13]:
PERMANOVA(
  df = dados,
  descritores = descritores,
  fatores = fatores,
  permutations = 9999,
  by = NULL
)


CENTROIDE 

Permutation test for adonis under reduced model
Permutation: free
Number of permutations: 9999

adonis2(formula = formula, data = dados, permutations = permutations, by = by)
         Df SumOfSqs      R2     F Pr(>F)    
Model    15    78.63 0.99532 906.6  1e-04 ***
Residual 64     0.37 0.00468                 
Total    79    79.00 1.00000                 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

LOGATTACKTIME 

Permutation test for adonis under reduced model
Permutation: free
Number of permutations: 9999

adonis2(formula = formula, data = dados, permutations = permutations, by = by)
         Df SumOfSqs      R2     F Pr(>F)    
Model    15   56.358 0.71339 10.62  1e-04 ***
Residual 64   22.642 0.28661                 
Total    79   79.000 1.00000                 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

CENTROIDE + LOGATTACKTIME 

Permutation test for adonis under reduced model
Permutation: free
Number of permutations:

# PERMDISP

In [14]:
PERMDISP <- function(df, descritores, fator, permutations = 9999) {
  
  dados <- prepDados(df, descritores)
  matriz_dist <- calcMatrizDistancia(dados, "euclidean")
  
  bd <- betadisper(matriz_dist, group = df[[fator]])
  
  # Testes
  printCabecalho(paste(toupper('ANOVA PARAMÉTRICA')))
  anova_disp <- anova(bd)
  print(anova_disp)

  printCabecalho(paste(toupper('TESTE POR PERMUTAÇÃO')))
  perm_disp <- permutest(bd, permutations = permutations)
  print(perm_disp)

  printCabecalho(paste(toupper('TUKEY HSD')))
  tuk_disp <- TukeyHSD(bd)
  print(tuk_disp)
}


In [15]:
PERMDISP(
  df = dados,
  descritores = descritores,
  fator = "metodo",
  permutations = 9999
)


ANOVA PARAMÉTRICA 

Analysis of Variance Table

Response: Distances
          Df  Sum Sq Mean Sq F value   Pr(>F)   
Groups     1  3.0447 3.04471  11.634 0.001029 **
Residuals 78 20.4138 0.26171                    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

TESTE POR PERMUTAÇÃO 


Permutation test for homogeneity of multivariate dispersions
Permutation: free
Number of permutations: 9999

Response: Distances
          Df  Sum Sq Mean Sq      F N.Perm Pr(>F)    
Groups     1  3.0447 3.04471 11.634   9999  1e-04 ***
Residuals 78 20.4138 0.26171                         
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

TUKEY HSD 

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = distances ~ group, data = df)

$group
                     diff      lwr       upr     p adj
natural-bending 0.3901739 0.162435 0.6179128 0.0010286



In [16]:
PERMDISP(
  df = dados,
  descritores = descritores,
  fator = "material",
  permutations = 9999
)


ANOVA PARAMÉTRICA 

Analysis of Variance Table

Response: Distances
          Df  Sum Sq Mean Sq F value Pr(>F)
Groups     1  0.2511 0.25106  0.7566 0.3871
Residuals 78 25.8818 0.33182               

TESTE POR PERMUTAÇÃO 


Permutation test for homogeneity of multivariate dispersions
Permutation: free
Number of permutations: 9999

Response: Distances
          Df  Sum Sq Mean Sq      F N.Perm Pr(>F)
Groups     1  0.2511 0.25106 0.7566   9999  0.396
Residuals 78 25.8818 0.33182                     

TUKEY HSD 

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = distances ~ group, data = df)

$group
                       diff        lwr       upr     p adj
plastico-madeira -0.1120413 -0.3684737 0.1443911 0.3870528



In [17]:
PERMDISP(
  df = dados,
  descritores = descritores,
  fator = "nota",
  permutations = 9999
)


ANOVA PARAMÉTRICA 

Analysis of Variance Table

Response: Distances
          Df  Sum Sq Mean Sq F value    Pr(>F)    
Groups     3  5.3265 1.77551  8.8044 4.415e-05 ***
Residuals 76 15.3262 0.20166                      
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

TESTE POR PERMUTAÇÃO 


Permutation test for homogeneity of multivariate dispersions
Permutation: free
Number of permutations: 9999

Response: Distances
          Df  Sum Sq Mean Sq      F N.Perm Pr(>F)    
Groups     3  5.3265 1.77551 8.8044   9999  4e-04 ***
Residuals 76 15.3262 0.20166                         
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

TUKEY HSD 

  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = distances ~ group, data = df)

$group
            diff         lwr       upr     p adj
E4-A4  0.5435863  0.17056161 0.9166109 0.0014768
F4-A4  0.3009469 -0.07207778 0.6739716 0.1562961
G4-A4  0.6781922  0.30516753 1.0512

### Auxiliar

In [18]:
colSignificancia <- function(resultados, coluna_p = "p_ajustado") {
  resultados$Significancia <- cut(
    resultados[[coluna_p]],
    breaks = c(0, 0.001, 0.01, 0.05, 1),
    labels = c("***", "**", "*", "n.s."),
    include.lowest = TRUE
  )
  resultados[order(resultados[[coluna_p]]), ]
}

# Post-hoc principal: PERMANOVA par-a-par

In [19]:
execPermanovaPar <- function(matriz_dist, df, fator, par, permutations) {
  indices <- df[[fator]] %in% par
  D_par <- as.dist(as.matrix(matriz_dist)[indices, indices])
  df_par <- df[indices, ]
  
  formula_str <- paste("D_par ~", fator)
  
  resultado <- adonis2(as.formula(formula_str), data = df_par, 
                       permutations = permutations, by = NULL)
  
  data.frame(
    Par = paste(par[1], "vs", par[2]),
    F = resultado$F[1],
    R2 = resultado$R2[1],
    p = resultado$`Pr(>F)`[1]
  )
}


In [20]:
posthocPermanova <- function(df, descritores, fator, permutations = 9999) {
  
  printCabecalho(paste("POST-HOC PAR-A-PAR:", toupper(fator)))
  
  dados_padronizados <- prepDados(df, descritores)
  matriz_dist <- calcMatrizDistancia(dados_padronizados, "euclidean")
  
  combinacoes <- combn(unique(df[[fator]]), 2, simplify = FALSE)
  resultados <- do.call(rbind, lapply(combinacoes, function(par) {
    execPermanovaPar(matriz_dist, df, fator, par, permutations)
  }))
  
  resultados$p_ajustado <- p.adjust(resultados$p, method = "holm")
  resultados <- colSignificancia(resultados, "p_ajustado")
  
  print(resultados)
}

In [21]:
posthocPermanova(
  df = dados,
  descritores = descritores,
  fator = "metodo",
  permutations = 9999
)


POST-HOC PAR-A-PAR: METODO 

                 Par        F        R2     p p_ajustado Significancia
1 bending vs natural 174.7491 0.6913935 1e-04      1e-04           ***


In [22]:
posthocPermanova(
  df = dados,
  descritores = descritores,
  fator = "material",
  permutations = 9999
)


POST-HOC PAR-A-PAR: MATERIAL 

                  Par         F         R2      p p_ajustado Significancia
1 madeira vs plastico 0.1919095 0.00245434 0.7892     0.7892          n.s.


In [23]:
posthocPermanova(
  df = dados,
  descritores = descritores,
  fator = "nota",
  permutations = 9999
)


POST-HOC PAR-A-PAR: NOTA 

       Par          F          R2      p p_ajustado Significancia
6 A4 vs F4 2.65489899 0.065303298 0.0863     0.5178          n.s.
1 E4 vs G4 0.06547484 0.001720059 0.9130     1.0000          n.s.
2 E4 vs A4 1.34917874 0.034287342 0.2457     1.0000          n.s.
3 E4 vs F4 1.27074982 0.032358685 0.2602     1.0000          n.s.
4 G4 vs A4 0.62343505 0.016141367 0.4985     1.0000          n.s.
5 G4 vs F4 1.23149914 0.031390570 0.2727     1.0000          n.s.


# Pós-hoc complementar em dispersão (se PERMDISP significativa)

In [24]:
posthocPermdisp <- function(df, descritores, fator, permutations = 9999) {
  
  printCabecalho(paste("POST-HOC DISPERSÃO:", toupper(fator)))
  
  dados_padronizados <- prepDados(df, descritores)
  matriz_dist <- calcMatrizDistancia(dados_padronizados, "euclidean")
  
  bd <- betadisper(matriz_dist, group = df[[fator]])
  
  # Tukey HSD
  distancias <- data.frame(grupo = df[[fator]], distancia = bd$distances)
  tukey_result <- TukeyHSD(aov(distancia ~ grupo, data = distancias))
  tukey_df <- as.data.frame(tukey_result$grupo)
  
  tabela <- data.frame(
    Par = rownames(tukey_df),
    Δdispersão = tukey_df$diff,
    IC_inf = tukey_df$lwr,
    IC_sup = tukey_df$upr,
    p_ajustado = tukey_df$`p adj`
  )
  
  tabela <- colSignificancia(tabela, "p_ajustado")
  
  print(tabela)
}

In [25]:
posthocPermdisp(
  df = dados,
  descritores = descritores,
  fator = "metodo",
  permutations = 9999
)


POST-HOC DISPERSÃO: METODO 

              Par Δdispersão   IC_inf    IC_sup  p_ajustado Significancia
1 natural-bending  0.3901739 0.162435 0.6179128 0.001028649            **
